### Tables

In [0]:
tables = [
    "end_to_end_retail.db_landing.tbl_customer_autoloader",
    "end_to_end_retail.db_landing.tbl_customer_taxid",
    "end_to_end_retail.db_landing.tbl_sales",
    "end_to_end_retail.db_landing.tbl_sales_oders_ordered_product",
    "end_to_end_retail.db_landing.tbl_source_files_customers",
    "end_to_end_retail.db_landing.tbl_source_files_sales",
    "end_to_end_retail.db_landing.tbl_source_files_sales_orders",
    "end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched",
    "end_to_end_retail.db_bronze.tbb_customer_changes_daily",
    "end_to_end_retail.db_bronze.tbb_northstar_outfitters_orders"
]

### Unity Catalog

#### Column Access Control

In [0]:
%sql
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.email_mask(column_value STRING)
RETURN CASE
  WHEN current_user() = 'user_one@company.com' THEN column_value
  ELSE sha2(email, 256)
END;

In [0]:
%sql
SELECT 
  customer_id,
  end_to_end_retail.governance_functions.email_mask(email) AS masked_email
FROM end_to_end_retail.db_landing.tbl_customer_autoloader
limit 5

In [0]:
%sql
-- create a SQL function for a simple column mask:
CREATE OR REPLACE FUNCTION end_to_end_retail.governance_functions.simple_mask(column_value STRING)
RETURN 
  case when is_account_group_member('group_example') then column_value else sha2(column_value, 256) END;

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE end_to_end_retail.db_bronze.tbl_customer_autoloader_masked (
    address STRING,
    city STRING,
    customer_id LONG,
    email STRING MASK end_to_end_retail.db_landing.simple_mask,
    name STRING,
    operation STRING,
    state STRING,
    timestamp DATE,
    zip_code STRING,
    _rescued_data STRING,
    file_path STRING,
    ingestion_date TIMESTAMP
)
""")

#### Row-level access control
- example

In [0]:
%sql
-- get the current user (for informational purposes)
SELECT current_user(), is_account_group_member('account_example');

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 10

In [0]:
%sql
CREATE OR REPLACE FUNCTION region_filter(region_param STRING) 
RETURN 
  is_account_group_member('account_example') or  
  region_param like "NY%";

In [0]:
%sql
SELECT region_filter('NY'), region_filter('GA')

In [0]:
%sql
ALTER TABLE end_to_end_retail.db_landing.tbl_customer_autoloader SET ROW FILTER region_filter ON (state);

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 15

In [0]:
%sql
ALTER TABLE end_to_end_retail.db_landing.tbl_customer_autoloader DROP ROW FILTER;

SELECT DISTINCT(state) FROM end_to_end_retail.db_landing.tbl_customer_autoloader;